# EXAMEN PRIMER BIMESTRE - RECUPERACION DE INFORMACION

**Nombre:** Joshua Daniel Menendez Farias

**Fecha:** 18-11-2026

In [7]:
# CARGA DEL DATASET DE SCIFACT 
from datasets import load_dataset
from preprocesamiento import preprocess
import pandas as pd
import numpy as np

# Cargar los datos de SciFact
print("Cargando SciFact desde Hugging Face...")
corpus_dataset = load_dataset("BeIR/scifact", "corpus")["corpus"]
queries_dataset = load_dataset("BeIR/scifact", "queries")["queries"]
qrels_dataset = load_dataset("BeIR/scifact-qrels")["test"] 

Cargando SciFact desde Hugging Face...


## Decisiones de Preprocesamiento del Corpus SciFact

Se implemento un pipeline de preprocesamiento tanto para los documentos del corpus como para las consultas entrantes, aplicando las siguientes etapas:

- Se convirtió todo el texto a minúsculas para evitar duplicidad de términos por variaciones tipográficas
- Se emplearon expresiones regulares para eliminar símbolos de puntuación, manteniendo los caracteres alfanuméricos 
- Se segmentó el texto continuo en unidades léxicas individuales (tokens) utilizando `word_tokenize` de la librería NLTK.
- Se filtraron las palabras vacías del inglés (`stopwords.words('english')`) y tokens de un solo carácter que carecen de significado semántico y añaden ruido innecesario a las matrices de términos.
- Se aplicó el algoritmo `PorterStemmer` de NLTK para reducir las palabras a su raíz gramatical 


In [ ]:
# Preprocesamiento y preparación de datos
docs_ids = []
textos_originales = {}
corpus_tokens = {}
metadata_docs = {}

print("Procesando e indexando el corpus...")

for doc in corpus_dataset:
    doc_id = doc["_id"]
    titulo = doc["title"]
    texto = doc["text"]
    texto_completo = f"{titulo} {texto}"
    
    # 1. Lista de IDs ordenados
    docs_ids.append(doc_id)
    
    # 2. Texto original completo para TF-IDF
    textos_originales[doc_id] = texto_completo
    
    # 3. Metadatos listos para la salida del servicio web JSON
    metadata_docs[doc_id] = {
        "title": titulo,
        "snippet": texto[:200] + "..." if len(texto) > 200 else texto
    }
    
    # 4. Tokens procesados para el índice BM25
    corpus_tokens[doc_id] = preprocess(texto_completo)

print(f"Total de documentos procesados: {len(docs_ids)}")

Procesando e indexando el corpus...
Total de documentos procesados: 5183


In [ ]:
# Preparación de consultas de prueba y qrels
# 1. Convertimos todos los IDs de los qrels a texto (String) para que coincidan con las queries
test_query_ids = set(str(qid) for qid in qrels_dataset["query-id"])

consultas_test = {}    # query-id -> texto original
consultas_tokens = {}  # query-id -> lista de tokens limpios

print("Procesando las consultas de prueba...")

# 2. Iteramos asegurando que el ID de la consulta también se trate como texto
for q in queries_dataset:
    q_id = str(q["_id"])
    if q_id in test_query_ids:
        consultas_test[q_id] = q["text"]
        consultas_tokens[q_id] = preprocess(q["text"])

print(f"Total de consultas de prueba procesadas y listas para evaluar: {len(consultas_test)}")

Procesando las consultas de prueba...
Total de consultas de prueba procesadas y listas para evaluar: 300


In [ ]:
# MODELO TF-IDF
from sklearn.feature_extraction.text import TfidfVectorizer

print("Construyendo modelo TF-IDF...")

# 1. Instanciar el vectorizador usando tu función de preprocesamiento
# token_pattern=None evita advertencias al usar un tokenizer personalizado
vectorizador_tfidf = TfidfVectorizer(tokenizer=preprocess, token_pattern=None, lowercase=False)

# 2. Alinear los textos en el orden exacto de docs_ids para mantener la correspondencia de índices
textos_lista = [textos_originales[doc_id] for doc_id in docs_ids]

# 3. Construir la matriz documento-término
matriz_tfidf = vectorizador_tfidf.fit_transform(textos_lista)

print(f"Dimensiones de la matriz TF-IDF: {matriz_tfidf.shape}")

Construyendo modelo TF-IDF...
Dimensiones de la matriz TF-IDF: (5183, 26038)


In [11]:
# MODELO BM25
import numpy as np
from collections import Counter

class BM25Index:
    def __init__(self, corpus_tokens, docs_ids, k1=1.5, b=0.75):
        self.k1 = k1
        self.b = b
        self.docs_ids = docs_ids
        self.N = len(docs_ids)
        
        # Estadísticas estáticas de la colección
        self.doc_len = {d: len(corpus_tokens[d]) for d in docs_ids}
        self.avgdl = sum(self.doc_len.values()) / self.N if self.N > 0 else 0.0
        self.tf = {d: Counter(corpus_tokens[d]) for d in docs_ids}
        
        self.df = Counter()
        for d in docs_ids:
            self.df.update(self.tf[d].keys())
            
        # IDF estándar de Robertson-Spärck Jones
        self.idf = {
            t: np.log((self.N - freq + 0.5) / (freq + 0.5) + 1.0)
            for t, freq in self.df.items()
        }

    def buscar(self, consulta_tokens, top_k=10):
        # Filtrar solo términos presentes en el vocabulario
        tokens_validos = [t for t in consulta_tokens if t in self.idf]
        if not tokens_validos:
            return []
        
        resultados = []
        for d in self.docs_ids:
            score = 0.0
            dl = self.doc_len[d]
            tf_doc = self.tf[d]
            
            for t in tokens_validos:
                f = tf_doc.get(t, 0)
                if f > 0:
                    numerador = f * (self.k1 + 1.0)
                    denominador = f + self.k1 * (1.0 - self.b + self.b * (dl / self.avgdl))
                    score += self.idf[t] * (numerador / denominador)
            
            if score > 0.0:
                resultados.append((d, float(score)))
                
        # Ordenar por puntuación descendente
        resultados.sort(key=lambda x: (-x[1], x[0]))
        return resultados[:top_k]

print("Construyendo índice BM25...")
bm25 = BM25Index(corpus_tokens, docs_ids, k1=1.5, b=0.75)
print("Índice BM25 construido exitosamente.")

Construyendo índice BM25...
Índice BM25 construido exitosamente.


In [ ]:
# Funciones de búsqueda y evaluación
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd
import numpy as np

# 1. Función de búsqueda rápida para TF-IDF
def buscar_tfidf(consulta_str, top_k=10):
    vec_q = vectorizador_tfidf.transform([consulta_str])
    scores = cosine_similarity(vec_q, matriz_tfidf).flatten()
    
    # Filtrar scores > 0 y ordenar
    indices_validos = np.where(scores > 0.0)[0]
    resultados = [(docs_ids[i], float(scores[i])) for i in indices_validos]
    resultados.sort(key=lambda x: (-x[1], x[0]))
    return resultados[:top_k]

# 2. Funciones de Evaluación a k=10
def precision_at_k(recuperados, relevantes_set, k=10):
    recuperados_k = recuperados[:k]
    if not recuperados_k: return 0.0
    rel_recuperados = [doc for doc in recuperados_k if doc in relevantes_set]
    return len(rel_recuperados) / k

def recall_at_k(recuperados, relevantes_set, k=10):
    if not relevantes_set: return 0.0
    recuperados_k = recuperados[:k]
    rel_recuperados = [doc for doc in recuperados_k if doc in relevantes_set]
    return len(rel_recuperados) / len(relevantes_set)

def ndcg_at_k(recuperados, relevantes_dict, k=10):
    recuperados_k = recuperados[:k]
    dcg = 0.0
    for i, doc in enumerate(recuperados_k):
        if doc in relevantes_dict:
            rel = relevantes_dict[doc]
            dcg += rel / np.log2(i + 2) # i empieza en 0, posición + 1 -> i + 2
            
    # Calcular el Ideal DCG (IDCG)
    relevancias_ideales = sorted(list(relevantes_dict.values()), reverse=True)[:k]
    idcg = 0.0
    for i, rel in enumerate(relevancias_ideales):
        idcg += rel / np.log2(i + 2)
        
    if idcg == 0.0: return 0.0
    return dcg / idcg

print("Funciones de búsqueda y métricas definidas correctamente.")

Funciones de búsqueda y métricas definidas correctamente.


In [ ]:
# Bucle de evaluación de consultas
print("Iniciando evaluación...")

# 1. Estructurar qrels para acceso O(1)
# qrels_dict[query_id] = {doc_id: score, ...}
qrels_dict = {}
for item in qrels_dataset:
    q_id = str(item['query-id'])
    doc_id = str(item['corpus-id'])
    score = int(item['score'])
    if score > 0: # Solo guardamos documentos que tienen relevancia positiva
        if q_id not in qrels_dict:
            qrels_dict[q_id] = {}
        qrels_dict[q_id][doc_id] = score

# 2. Diccionarios para almacenar los resultados de cada consulta
metricas_tfidf = {'P@10': [], 'R@10': [], 'NDCG@10': []}
metricas_bm25 = {'P@10': [], 'R@10': [], 'NDCG@10': []}
consultas_evaluadas = []

# 3. Bucle de Evaluación
for q_id, texto_original in consultas_test.items():
    if q_id not in qrels_dict:
        continue # Saltamos consultas sin documentos relevantes en los qrels
        
    relevantes_dict = qrels_dict[q_id]
    relevantes_set = set(relevantes_dict.keys())
    tokens_limpios = consultas_tokens[q_id]
    consultas_evaluadas.append(q_id)
    
    # Obtener IDs del Top 10 
    top_tfidf = [doc for doc, _ in buscar_tfidf(texto_original, top_k=10)]
    top_bm25 = [doc for doc, _ in bm25.buscar(tokens_limpios, top_k=10)]
    
    # Evaluar TF-IDF
    metricas_tfidf['P@10'].append(precision_at_k(top_tfidf, relevantes_set))
    metricas_tfidf['R@10'].append(recall_at_k(top_tfidf, relevantes_set))
    metricas_tfidf['NDCG@10'].append(ndcg_at_k(top_tfidf, relevantes_dict))
    
    # Evaluar BM25
    metricas_bm25['P@10'].append(precision_at_k(top_bm25, relevantes_set))
    metricas_bm25['R@10'].append(recall_at_k(top_bm25, relevantes_set))
    metricas_bm25['NDCG@10'].append(ndcg_at_k(top_bm25, relevantes_dict))

# 4. Presentación de la tabla comparativa
tabla_resultados = pd.DataFrame({
    'Modelo': ['TF-IDF + Coseno', 'Okapi BM25'],
    'Precision@10': [np.mean(metricas_tfidf['P@10']), np.mean(metricas_bm25['P@10'])],
    'Recall@10': [np.mean(metricas_tfidf['R@10']), np.mean(metricas_bm25['R@10'])],
    'NDCG@10': [np.mean(metricas_tfidf['NDCG@10']), np.mean(metricas_bm25['NDCG@10'])]
})

print(f"Evaluación finalizada sobre {len(consultas_evaluadas)} consultas.")
display(tabla_resultados.round(4))

Iniciando evaluación...
Evaluación finalizada sobre 300 consultas.


,Modelo,Precision@10,Recall@10,NDCG@10
0,TF-IDF + Coseno,0.0820,0.7476,0.5897
1,Okapi BM25,0.0917,0.8287,0.6880



A partir de los resultados obtenidos tras evaluar el conjunto de prueba test de SciFact compuesto por 300 consultas y sus respectivos juicios de relevancia qrels, se presentan las siguientes conclusiones técnicas:

  - El modelo probabilístico BM25 supera consistentemente a TF-IDF con similitud Coseno en las tres métricas evaluadas a k=10 (Precision@10, Recall@10 y NDCG@10). 
  - Esto se debe fundamentalmente a la arquitectura matemática subyacente de BM25 frente al enfoque puramente vectorial de TF-IDF.

**Conclusión de las Métricas de Ranking ($NDCG@10$):**

  - Con los resultados de NDCG@10 se demuestra que BM25 recupera una mayor cantidad de documentos verdaderamente relevantes dentro del top 10 (Recall@10 superior), adicionalmente, los posiciona en los rangos superiores (puestos 1 a 3) con mayor precisión que el modelo vectorial de espacio de Coseno.

In [ ]:
# Construcción del índice invertido para BM25
inverted_index = {} # termino -> {doc_id: term_frequency}

for doc_id, tokens in corpus_tokens.items():
    conteos = Counter(tokens)
    for termino, freq in conteos.items():
        if termino not in inverted_index:
            inverted_index[termino] = {}
        inverted_index[termino][doc_id] = freq

print(f"Tamaño del vocabulario: {len(inverted_index)} términos")
print(f"Ejemplo de postings para 'cell': {list(inverted_index.get('cell', {}).items())[:5]}")
print(f"\nIDs de consultas evaluadas ({len(consultas_evaluadas)}): {consultas_evaluadas[:15]} ... (hasta {consultas_evaluadas[-1]})")

Tamaño del vocabulario: 26038 términos
Ejemplo de postings para 'cell': [('5836', 5), ('18670', 3), ('19238', 4), ('33370', 9), ('36474', 6)]

IDs de consultas evaluadas (300): ['1', '3', '5', '13', '36', '42', '48', '49', '50', '51', '53', '54', '56', '57', '70'] ... (hasta 1395)


In [ ]:
# Guardado de artefactos para el servicio web
import pickle

artefactos = {
    "docs_ids": docs_ids,
    "metadata": metadata_docs,
    "vectorizador": vectorizador_tfidf,
    "matriz_tfidf": matriz_tfidf,
    "bm25": {
        "k1": bm25.k1,
        "b": bm25.b,
        "avgdl": bm25.avgdl,
        "doc_len": bm25.doc_len,
        "idf": bm25.idf,
        "inverted_index": inverted_index
    }
}

with open("ir_artifacts.pkl", "wb") as f:
    pickle.dump(artefactos, f)

print("Archivo 'ir_artifacts.pkl' generado correctamente.")

Archivo 'ir_artifacts.pkl' generado correctamente.


- Para las consultas en la web, los modelos se llaman:
    - model": "bm25",
    - "model": "tfidf"